In [2]:
import os
import sys
import torch
import torch.nn as nn
from gymnasium import spaces
from luxai_s3.params import EnvParams
from luxai_s3.wrappers import LuxAIS3GymEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize

# Add the project root directory to the Python path
sys.path.append(os.path.abspath(".."))
from common.helper import EnhancedTensorboardCallback, custom_env_check, linear_schedule, make_env
from wrappers.mappo_wrapper import SB3LuxEnvMAPPO
from common.environment import GameConstants

## Check if the environment is compatible with SB3

In [3]:
# Create the base environment
base_env = LuxAIS3GymEnv()
# Create environment parameters
env_params = EnvParams(map_type=0, max_steps_in_match=100)

# Apply our wrapper with explicit player_id and opponent strategy
wrapped_env = SB3LuxEnvMAPPO(base_env, player_id='player_0', opponent_strategy='random')

# Create the environment
env = make_env(env_params, wrapped_env, seed=367)  # Using a fixed seed for reproducibility

# Print observation space shape
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

check_env(env)
print("Environment check passed successfully!")

Observation space: Dict('env_cfg_map_height': Box(0, 24, (1,), int32), 'env_cfg_map_width': Box(0, 24, (1,), int32), 'env_cfg_max_steps_in_match': Box(0, 100, (1,), int32), 'env_cfg_unit_move_cost': Box(0, 100, (1,), int32), 'env_cfg_unit_sap_cost': Box(0, 100, (1,), int32), 'env_cfg_unit_sap_range': Box(0, 100, (1,), int32), 'map_features_energy': Box(-1, 20, (24, 24), int8), 'map_features_tile_type': Box(-1, 2, (24, 24), int8), 'match_steps': Box(0, 100, (1,), int32), 'relic_nodes': Box(-1, 23, (6, 2), int32), 'relic_nodes_mask': Box(0, 1, (6,), int8), 'remainingOverageTime': Box(0, 1000, (1,), int32), 'sensor_mask': Box(0, 1, (24, 24), int8), 'steps': Box(0, 100, (1,), int32), 'team_points': Box(0, 1000, (2,), int32), 'team_wins': Box(0, 1000, (2,), int32), 'units_energy': Box(0, 400, (2, 16, 1), int32), 'units_mask': Box(0, 1, (2, 16), int8), 'units_position': Box(-1, 575, (2, 16, 2), int32))
Action space: MultiDiscrete([6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6])


/opt/anaconda3/envs/inf367_3/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation map_features_energy has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/opt/anaconda3/envs/inf367_3/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation map_features_tile_type has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/opt/anaconda3/envs/inf367_3/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation relic_nodes has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or u

Environment check passed successfully!


#### Custom enviorment check 🤖

In [4]:
# custom_env_check(env, 2)

### Train the base PPO model

In [5]:
class LuxFeaturesExtractor(BaseFeaturesExtractor):
    """
    Features extractor for the Lux AI observation space.
    """
    def __init__(self, observation_space: spaces.Dict, features_dim: int = 512):
        super().__init__(observation_space, features_dim)
        
        # Define extractors for each type of observation
        self.unit_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(GameConstants.NUM_TEAMS * GameConstants.MAX_UNITS * 2, 128),
            nn.ReLU()
        )
        
        self.energy_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(GameConstants.NUM_TEAMS * GameConstants.MAX_UNITS * 1, 64),
            nn.ReLU()
        )
        
        self.mask_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(GameConstants.NUM_TEAMS * GameConstants.MAX_UNITS, 64),
            nn.ReLU()
        )
        
        # CNN for spatial observations (maps)
        self.cnn_encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),  # 3 channels: sensor_mask, tile_type, energy
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * (GameConstants.MAP_WIDTH//4) * (GameConstants.MAP_HEIGHT//4), 128),
            nn.ReLU()
        )
        
        # For relic nodes
        self.relic_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(GameConstants.MAX_RELIC_NODES * 2 + GameConstants.MAX_RELIC_NODES, 64),
            nn.ReLU()
        )
        
        # For other game state info
        self.game_state_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * GameConstants.NUM_TEAMS + 9, 64),  # team_points, team_wins, steps, etc.
            nn.ReLU()
        )
        
        # Final layer to combine all features
        self.final_layer = nn.Sequential(
            nn.Linear(128 + 64 + 64 + 128 + 64 + 64, features_dim),
            nn.ReLU()
        )
    
    def forward(self, observations):
        # Process unit positions
        unit_features = self.unit_encoder(observations["units_position"])
        
        # Process unit energy
        energy_features = self.energy_encoder(observations["units_energy"])
        
        # Process unit masks
        mask_features = self.mask_encoder(observations["units_mask"])
        
        # Process spatial features using CNN
        # Stack the spatial observations
        spatial_input = torch.cat([
            observations["sensor_mask"].unsqueeze(1),
            observations["map_features_tile_type"].unsqueeze(1),
            observations["map_features_energy"].unsqueeze(1)
        ], dim=1)
        spatial_features = self.cnn_encoder(spatial_input)
        
        # Process relic nodes
        relic_input = torch.cat([
            observations["relic_nodes"].flatten(1),
            observations["relic_nodes_mask"].flatten(1)
        ], dim=1)
        relic_features = self.relic_encoder(relic_input)
        
        # Process other game state info
        game_state_input = torch.cat([
            observations["team_points"].flatten(1),
            observations["team_wins"].flatten(1),
            observations["steps"].flatten(1),
            observations["match_steps"].flatten(1),
            observations["remainingOverageTime"].flatten(1),
            observations["env_cfg_map_width"].flatten(1),
            observations["env_cfg_map_height"].flatten(1),
            observations["env_cfg_max_steps_in_match"].flatten(1),
            observations["env_cfg_unit_move_cost"].flatten(1),
            observations["env_cfg_unit_sap_cost"].flatten(1),
            observations["env_cfg_unit_sap_range"].flatten(1)
        ], dim=1)
        game_state_features = self.game_state_encoder(game_state_input)
        
        # Combine all features
        combined_features = torch.cat([
            unit_features, 
            energy_features, 
            mask_features, 
            spatial_features, 
            relic_features, 
            game_state_features
        ], dim=1)
        
        return self.final_layer(combined_features)
    
class RNNPolicy(ActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super(RNNPolicy, self).__init__(
            *args,
            **kwargs,
        )
        
        # Define LSTM state size
        self.lstm_state_size = self.features_dim
        
        # Add LSTM layers after the features extractor
        self.lstm = nn.LSTM(
            input_size=self.features_dim,
            hidden_size=self.features_dim,
            num_layers=1,
            batch_first=True
        )
        
        # Initialize hidden states
        self.hidden = None
    
    def _get_lstm_features(self, features):
        # Reshape for LSTM (add time dimension if not already there)
        batch_size = features.shape[0]
        if len(features.shape) == 2:
            features = features.unsqueeze(1)  # Add time dimension
        
        # Initialize hidden state if needed
        if self.hidden is None or self.hidden[0].shape[1] != batch_size:
            self.hidden = (
                torch.zeros(1, batch_size, self.lstm_state_size).to(self.device),
                torch.zeros(1, batch_size, self.lstm_state_size).to(self.device)
            )
        
        # Pass through LSTM
        lstm_out, self.hidden = self.lstm(features, self.hidden)
        return lstm_out.squeeze(1)  # Remove time dimension
    
    def forward(self, obs, deterministic=False):
        # Extract features
        features = self.extract_features(obs)
        
        # Process through LSTM
        lstm_features = self._get_lstm_features(features)
        
        # Get action distribution and value estimate
        latent_pi, latent_vf = self.mlp_extractor(lstm_features)
        
        distribution = self._get_action_dist_from_latent(latent_pi)
        actions = distribution.get_actions(deterministic=deterministic)
        log_probs = distribution.log_prob(actions)
        values = self.value_net(latent_vf)
        
        return actions, values, log_probs
    
    def _predict(self, observation, deterministic=False):
        # Override to reset hidden state when predicting
        self.hidden = None
        return super()._predict(observation, deterministic)

In [6]:
models_dir = "../ppo_lux_model_mappohr/"
SAVE_FREQUENCY = 1000

envs = SubprocVecEnv(
    [lambda: make_env(env_params, wrapped_env, seed=i) for i in range(8)]
)
envs = VecNormalize(envs, norm_obs=True, norm_reward=True)

# Create callbacks
enchared_tb_callback = EnhancedTensorboardCallback()
checkpoint_callback = CheckpointCallback(
    save_freq=SAVE_FREQUENCY, save_path=models_dir, name_prefix="ppo_lux_model_mappohr"
)

# Combine callbacks
callbacks = CallbackList([enchared_tb_callback, checkpoint_callback])

mps_device = None
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")
else:
    mps_device = torch.device("mps")

    print("MPS available and enabled.")
    


# Create PPO model
model = PPO(
    RNNPolicy,  # Custom RNN policy
    envs,
    verbose=1,
    learning_rate=linear_schedule(5e-4, 1e-4),
    n_steps=2048,        
    batch_size=128, 
    n_epochs=5,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    vf_coef=0.25,
    ent_coef=3.0,
    clip_range_vf=0.2,
    tensorboard_log="./ppo_lux_tensorboard/",
    device=mps_device,
    policy_kwargs=dict(
        features_extractor_class=LuxFeaturesExtractor,
        features_extractor_kwargs=dict(features_dim=512),
        net_arch=dict(pi=[128, 64], vf=[128, 64]),
        activation_fn=torch.nn.ReLU
    )
)

# Train the model with callbacks
model.learn(total_timesteps=1000000, callback=callbacks)

# Save final model
model.save("ppo_lux_model_mappohr")

MPS available and enabled.
Using mps device
Logging to ./ppo_lux_tensorboard/PPO_6
-------------------------------------------------
| lux/                               |          |
|    collision_detected              | 0        |
|    energy_collected                | 4        |
|    final_mappo_reward              | 22.1     |
|    final_reward                    | 16.1     |
|    map_coverage                    | 5.73     |
|    mappo_reward                    | 0        |
|    new_tiles_revealed              | 0        |
|    point_reward                    | 0        |
|    points_earned                   | 0        |
|    relic_control_streak            | 0        |
|    relic_point_tiles_found         | 0        |
|    replan_decisions                | 0        |
|    rule_reward                     | 16.1     |
|    sap_actions_taken               | 3        |
|    sap_reward                      | 0        |
|    static_planner_bonus_sap_reward | 6        |
|    total_energy

KeyboardInterrupt: 